In [1]:
# --- Notebook Test Engine: injected setup (auto-generated) ---
import os as _os
for _k in ('AWS_ACCESS_KEY_ID','AWS_SECRET_ACCESS_KEY','AWS_SESSION_TOKEN','AWS_CREDENTIAL_EXPIRATION'):
    _os.environ.pop(_k, None)
_os.environ.setdefault('AWS_DEFAULT_REGION', 'us-west-2')
_os.environ.setdefault('AWS_REGION', 'us-west-2')
_nte_role = 'arn:aws:iam::674622101542:role/NotebookTestEngine-ProcessingJobRole'
try:
    from sagemaker.core.helper import session_helper as _sh
    _sh.get_execution_role = lambda *a, **k: _nte_role
except Exception:
    pass
try:
    _nte_cf = _os.environ.get('AWS_SHARED_CREDENTIALS_FILE')
    if _nte_cf and _os.path.exists(_nte_cf):
        import configparser as _cfp, datetime as _dt
        import boto3 as _b3, botocore.session as _bcs
        from botocore.credentials import RefreshableCredentials as _RC
        _nte_prof = _os.environ.get('AWS_PROFILE', 'default')
        def _nte_load():
            _cp = _cfp.ConfigParser(); _cp.read(_nte_cf)
            _s = _cp[_nte_prof]
            _tok = _s.get('aws_session_token')
            if not _tok:
                raise RuntimeError('nte: no session token in creds file')
            return {'access_key': _s['aws_access_key_id'],
                    'secret_key': _s['aws_secret_access_key'],
                    'token': _tok,
                    'expiry_time': (_dt.datetime.now(_dt.timezone.utc)
                                    + _dt.timedelta(minutes=9)).isoformat()}
        _nte_creds = _RC.create_from_metadata(metadata=_nte_load(),
                                              refresh_using=_nte_load,
                                              method='nte-shared-file')
        def _nte_get_credentials(self, *a, **k):
            # Honor sessions that were handed explicit credentials
            # (e.g. notebooks that assume_role into other accounts):
            # clobbering them would silently run every cross-account
            # client as the ambient identity. Bare/SDK-created sessions
            # (no explicit creds) still get the refreshable shared-file
            # creds so long-running waits survive credential rotation.
            _ex = getattr(self, '_credentials', None)
            if _ex is not None:
                return _ex
            return _nte_creds
        _bcs.Session.get_credentials = _nte_get_credentials
        _b3.setup_default_session(botocore_session=_bcs.get_session())
        print('nte: refreshable shared-file creds installed')
except Exception as _e:
    print('nte: refreshable-creds shim skipped:', _e)
try:
    from sagemaker.core.resources import ModelPackageGroup as _NteMPG
    _nte_mpg_create = _NteMPG.create
    def _nte_mpg_get_or_create(*_a, **_k):
        try:
            return _nte_mpg_create(*_a, **_k)
        except Exception as _e2:
            if 'already exists' in str(_e2).lower():
                _name = _k.get('model_package_group_name') or (_a[0] if _a else None)
                return _NteMPG.get(_name)
            raise
    _NteMPG.create = staticmethod(_nte_mpg_get_or_create)
    print('nte: ModelPackageGroup.create is now get-or-create')
except Exception as _e:
    print('nte: ModelPackageGroup idempotency shim skipped:', _e)


nte: ModelPackageGroup.create is now get-or-create


# SageMaker V3 Inference Pipeline - ModelBuilder vs Core

This notebook demonstrates how to create and deploy an **inference pipeline** in SageMaker V3. An inference pipeline chains multiple containers together, where the output of one container becomes the input to the next.

### Prerequisites
Note: Ensure you have sagemaker and ipywidgets installed in your environment. The ipywidgets package is required to monitor endpoint deployment progress in Jupyter notebooks.

## What You'll Learn

1. Train models using `ModelTrainer` (high-level training API)
2. Package inference code with model artifacts using `repack_model`
3. Create multi-container pipeline models with `Model.create()`
4. Deploy pipelines using both low-level APIs and `ModelBuilder`

## Pipeline Architecture

```
Raw Data → [SKLearn: StandardScaler] → Scaled Data → [XGBoost: Classifier] → Predictions
```

- **Container 1 (Preprocessing)**: SKLearn StandardScaler normalizes input features
- **Container 2 (Inference)**: XGBoost binary classifier predicts outcomes

## Why Use Inference Pipelines?

- **Separation of concerns**: Preprocessing and inference logic in separate containers
- **Reusability**: Same preprocessing can be used with different models
- **Scalability**: Each container can be optimized independently
- **Maintainability**: Update one component without affecting others

---
## Step 1: Setup and Data Preparation

We start by importing the required modules and creating synthetic data for our binary classification task. The data has features at different scales to demonstrate the value of preprocessing.

In [2]:
import uuid
import os
import tempfile
import numpy as np
import pandas as pd
import boto3

from sagemaker.core.resources import Model, Endpoint, EndpointConfig
from sagemaker.core.shapes import ContainerDefinition, InferenceExecutionConfig, ProductionVariant
from sagemaker.core.image_uris import retrieve
from sagemaker.core.utils import repack_model
from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.train.model_trainer import ModelTrainer
from sagemaker.train.configs import SourceCode, InputData
from sagemaker.serve import ModelBuilder

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Fetched defaults config from location: /opt/ml/processing/input/sm_config.yaml


In [3]:
# Initialize session
sagemaker_session = Session()
role = get_execution_role()
region = sagemaker_session.boto_region_name
bucket = sagemaker_session.default_bucket()
unique_id = str(uuid.uuid4())[:8]
prefix = f"inference-pipeline-v3/{unique_id}"

print(f"Region: {region}")
print(f"Bucket: {bucket}")
print(f"Prefix: {prefix}")

Region: us-west-2
Bucket: sagemaker-us-west-2-674622101542
Prefix: inference-pipeline-v3/5e35f9a2


In [4]:
# Generate synthetic data
np.random.seed(42)
n_samples = 1000

feature1 = np.random.normal(100, 15, n_samples)
feature2 = np.random.normal(50, 10, n_samples)
feature3 = np.random.normal(0.5, 0.1, n_samples)
feature4 = np.random.normal(1000, 200, n_samples)
target = ((feature1 > 100) & (feature2 > 50) | (feature4 > 1100)).astype(int)

df = pd.DataFrame({
    'feature1': feature1, 'feature2': feature2,
    'feature3': feature3, 'feature4': feature4, 'target': target
})

train_df = df[:800]
test_df = df[800:]

# Upload training data
data_dir = tempfile.mkdtemp()
train_file = os.path.join(data_dir, 'train.csv')
train_df.to_csv(train_file, index=False, header=False)

s3_client = boto3.client('s3', region_name='us-west-2')
train_s3_key = f"{prefix}/data/train.csv"
s3_client.upload_file(train_file, bucket, train_s3_key)
train_data_uri = f"s3://{bucket}/{train_s3_key}"
print(f"Training data: {train_data_uri}")

Training data: s3://sagemaker-us-west-2-674622101542/inference-pipeline-v3/5e35f9a2/data/train.csv


---
## Step 2: Train SKLearn Model with ModelTrainer

`ModelTrainer` is the V3 high-level API for training. It simplifies job creation compared to the low-level `TrainingJob.create()` API.

**Key components:**
- `SourceCode`: Points to your training script and source directory
- `InputData`: Defines training data channels
- The training script only needs training logic - inference code is added separately later

In [5]:
# Create SKLearn training script (training only - no inference functions)
sklearn_source_dir = tempfile.mkdtemp()

sklearn_train_script = '''import argparse, os, joblib
import pandas as pd
from sklearn.preprocessing import StandardScaler

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--model-dir", type=str, default=os.environ.get("SM_MODEL_DIR", "/opt/ml/model"))
    parser.add_argument("--train", type=str, default=os.environ.get("SM_CHANNEL_TRAIN", "/opt/ml/input/data/train"))
    args = parser.parse_args()
    
    train_files = [os.path.join(args.train, f) for f in os.listdir(args.train) if f.endswith(".csv")]
    df = pd.concat([pd.read_csv(f, header=None) for f in train_files])
    X = df.iloc[:, :4].values
    
    scaler = StandardScaler()
    scaler.fit(X)
    
    os.makedirs(args.model_dir, exist_ok=True)
    joblib.dump(scaler, os.path.join(args.model_dir, "model.joblib"))
    print(f"Model saved to {args.model_dir}")
'''

with open(os.path.join(sklearn_source_dir, 'train.py'), 'w') as f:
    f.write(sklearn_train_script)

print(f"SKLearn training script: {sklearn_source_dir}")

SKLearn training script: /home/model-server/tmp/tmp0fe3t6bp


In [6]:
# Get SKLearn training image
sklearn_training_image = retrieve(
    framework="sklearn", region=region, version="1.4-2", py_version="py3"
)
print(f"SKLearn training image: {sklearn_training_image}")

[07/20/26 07:13:28] INFO     Defaulting to only supported image scope: cpu.                       ]8;id=16059236;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=16059237;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/image_uris.py#539\539]8;;\

SKLearn training image: 246618743249.dkr.ecr.us-west-2.amazonaws.com/sagemaker-scikit-learn:1.4-2-cpu-py3


In [7]:
# Train SKLearn model using ModelTrainer
sklearn_trainer = ModelTrainer(
    training_image=sklearn_training_image,
    source_code=SourceCode(
        source_dir=sklearn_source_dir,
        entry_script="train.py"
    ),
    base_job_name="sklearn-preprocess",
    role=role,
    sagemaker_session=sagemaker_session
)

sklearn_trainer.train(
    input_data_config=[InputData(channel_name="train", data_source=train_data_uri)]
)

sklearn_model_uri = sklearn_trainer._latest_training_job.model_artifacts.s3_model_artifacts
print(f"SKLearn model artifacts: {sklearn_model_uri}")

[07/20/26 07:13:29] INFO     Cannot simulate policies for                                  ]8;id=16059244;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=16059245;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#363\363]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (access denied); permission verdict unknown.                                 

                    WARNING  Could not verify permissions for role                         ]8;id=16059251;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=16059252;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#585\585]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (caller lacks iam:SimulatePrincipalPolicy).                                  
                             Proceeding with it. If the operation later fails with an                              
                             access-denied error, ensure the role has the required                                 
                             permissions for 'training' (see                                                       
                             IamRoleResolver().get_required_actions('training')) or create                         
                             a dedicated role via                                                                  
                             IamRoleResolver().create_execution_role(role_type='training')                         
                             .                                                                                     

                    INFO     Compute not provided. Using default:                                   ]8;id=16059259;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=16059260;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#141\141]8;;\
                             instance_type='ml.m5.xlarge' instance_count=1 volume_size_in_gb=30                    
                             volume_kms_key_id=None keep_alive_period_in_seconds=None                              
                             instance_groups=None training_plan_arn=None                                           
                             instance_placement_config=None enable_managed_spot_training=None                      

                    INFO     StoppingCondition not provided. Using default:                         ]8;id=16059266;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=16059267;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#167\167]8;;\
                             max_runtime_in_seconds=3600 max_wait_time_in_seconds=None                             
                             max_pending_time_in_seconds=None                                                      

                    INFO     OutputDataConfig not provided. Using default:                          ]8;id=16059273;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=16059274;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#192\192]8;;\
                             s3_output_path='s3://sagemaker-us-west-2-674622101542/sklearn-preproce                
                             ss' kms_key_id=None compression_type='GZIP'                                           

...

                    INFO     sklearn-preprocess-20260720071330/algo-1-1784531662:                ]8;id=16059823;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=16059824;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#31585\31585]8;;\
                             Model saved to /opt/ml/model                                                          

                    INFO     sklearn-preprocess-20260720071330/algo-1-1784531662:                ]8;id=16059829;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=16059830;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#31585\31585]8;;\
                             ++ echo 'Training Container Execution Completed'                                      

                    INFO     sklearn-preprocess-20260720071330/algo-1-1784531662:                ]8;id=16059835;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=16059836;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#31585\31585]8;;\
                             Training Container Execution Completed                                                

[07/20/26 07:15:33] INFO     Final Resource Status: Completed                                    ]8;id=16059842;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=16059843;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#31591\31591]8;;\

SKLearn model artifacts: s3://sagemaker-us-west-2-674622101542/sklearn-preprocess/sklearn-preprocess-20260720071330/output/model.tar.gz


---
## Step 3: Train XGBoost Model with ModelTrainer

We train an XGBoost classifier using the same `ModelTrainer` pattern. Note that we pass hyperparameters directly to the trainer.

In [8]:
# Create XGBoost training script
xgboost_source_dir = tempfile.mkdtemp()

xgboost_train_script = '''import argparse, os
import pandas as pd
import xgboost as xgb

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--model-dir", type=str, default=os.environ.get("SM_MODEL_DIR", "/opt/ml/model"))
    parser.add_argument("--train", type=str, default=os.environ.get("SM_CHANNEL_TRAIN", "/opt/ml/input/data/train"))
    parser.add_argument("--num-round", type=int, default=100)
    parser.add_argument("--max-depth", type=int, default=5)
    parser.add_argument("--eta", type=float, default=0.2)
    args = parser.parse_args()
    
    train_files = [os.path.join(args.train, f) for f in os.listdir(args.train) if f.endswith(".csv")]
    df = pd.concat([pd.read_csv(f, header=None) for f in train_files])
    X, y = df.iloc[:, :4].values, df.iloc[:, 4].values
    
    dtrain = xgb.DMatrix(X, label=y)
    params = {"max_depth": args.max_depth, "eta": args.eta, "objective": "binary:logistic"}
    model = xgb.train(params, dtrain, num_boost_round=args.num_round)
    
    os.makedirs(args.model_dir, exist_ok=True)
    model.save_model(os.path.join(args.model_dir, "xgboost-model"))
    print(f"Model saved to {args.model_dir}")
'''

with open(os.path.join(xgboost_source_dir, 'train.py'), 'w') as f:
    f.write(xgboost_train_script)

print(f"XGBoost training script: {xgboost_source_dir}")

XGBoost training script: /home/model-server/tmp/tmphqyq9rcc


In [9]:
# Get XGBoost training image
xgboost_training_image = retrieve(
    framework="xgboost", region=region, version="3.0-5",
)
print(f"XGBoost training image: {xgboost_training_image}")

[07/20/26 07:15:34] INFO     Ignoring unnecessary instance type: None.                            ]8;id=16059849;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=16059850;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/image_uris.py#535\535]8;;\

XGBoost training image: 246618743249.dkr.ecr.us-west-2.amazonaws.com/sagemaker-xgboost:3.0-5


In [10]:
# Train XGBoost model using ModelTrainer
xgboost_trainer = ModelTrainer(
    training_image=xgboost_training_image,
    source_code=SourceCode(
        source_dir=xgboost_source_dir,
        entry_script="train.py"
    ),
    hyperparameters={
        "num-round": 100,
        "max-depth": 5,
        "eta": 0.2
    },
    base_job_name="xgboost-classifier",
    role=role,
    sagemaker_session=sagemaker_session
)

xgboost_trainer.train(
    input_data_config=[InputData(channel_name="train", data_source=train_data_uri)]
)

xgboost_model_uri = xgboost_trainer._latest_training_job.model_artifacts.s3_model_artifacts
print(f"XGBoost model artifacts: {xgboost_model_uri}")

                    INFO     Cannot simulate policies for                                  ]8;id=16059855;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=16059856;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#363\363]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (access denied); permission verdict unknown.                                 

                    WARNING  Could not verify permissions for role                         ]8;id=16059861;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=16059862;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#585\585]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (caller lacks iam:SimulatePrincipalPolicy).                                  
                             Proceeding with it. If the operation later fails with an                              
                             access-denied error, ensure the role has the required                                 
                             permissions for 'training' (see                                                       
                             IamRoleResolver().get_required_actions('training')) or create                         
                             a dedicated role via                                                                  
                             IamRoleResolver().create_execution_role(role_type='training')                         
                             .                                                                                     

                    INFO     Compute not provided. Using default:                                   ]8;id=16059867;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=16059868;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#141\141]8;;\
                             instance_type='ml.m5.xlarge' instance_count=1 volume_size_in_gb=30                    
                             volume_kms_key_id=None keep_alive_period_in_seconds=None                              
                             instance_groups=None training_plan_arn=None                                           
                             instance_placement_config=None enable_managed_spot_training=None                      

                    INFO     StoppingCondition not provided. Using default:                         ]8;id=16059873;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=16059874;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#167\167]8;;\
                             max_runtime_in_seconds=3600 max_wait_time_in_seconds=None                             
                             max_pending_time_in_seconds=None                                                      

                    INFO     OutputDataConfig not provided. Using default:                          ]8;id=16059879;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=16059880;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#192\192]8;;\
                             s3_output_path='s3://sagemaker-us-west-2-674622101542/xgboost-classifi                
                             er' kms_key_id=None compression_type='GZIP'                                           

...

                    INFO     xgboost-classifier-20260720071535/algo-1-1784531774:                ]8;id=16060473;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=16060474;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#31585\31585]8;;\
                             Model saved to /opt/ml/model                                                          

                    INFO     xgboost-classifier-20260720071535/algo-1-1784531774:                ]8;id=16060479;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=16060480;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#31585\31585]8;;\
                             ++ echo 'Training Container Execution Completed'                                      

                    INFO     xgboost-classifier-20260720071535/algo-1-1784531774:                ]8;id=16060485;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=16060486;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#31585\31585]8;;\
                             Training Container Execution Completed                                                

[07/20/26 07:17:48] INFO     Final Resource Status: Completed                                    ]8;id=16060491;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=16060492;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#31591\31591]8;;\

XGBoost model artifacts: s3://sagemaker-us-west-2-674622101542/xgboost-classifier/xgboost-classifier-20260720071535/output/model.tar.gz


---
## Step 4: Create Inference Scripts and Repack Models

Training produces model artifacts (e.g., `model.tar.gz`) but these don't include inference code. The `repack_model` utility:

1. Downloads the original model artifacts from S3
2. Extracts them to a temporary directory
3. Adds your inference script to a `code/` subdirectory
4. Re-packages and uploads to S3

**Important for pipelines:** The `output_fn` must return a tuple `(data, content_type)` to explicitly set the content type passed to the next container. Without this, intermediate containers receive `application/json` as the default accept type.

In [11]:
# Create SKLearn inference script
sklearn_inference_dir = tempfile.mkdtemp()

sklearn_inference_script = '''import joblib, os
import numpy as np

def model_fn(model_dir):
    return joblib.load(os.path.join(model_dir, "model.joblib"))

def input_fn(request_body, request_content_type):
    if request_content_type == "text/csv":
        return np.array([[float(x) for x in line.split(",")] for line in request_body.strip().split("\\n")])
    raise ValueError(f"Unsupported content type: {request_content_type}")

def predict_fn(input_data, model):
    return model.transform(input_data)

def output_fn(prediction, accept):
    # Always return CSV with explicit content-type for pipeline compatibility
    csv_output = "\\n".join([",".join([str(x) for x in row]) for row in prediction])
    return csv_output, "text/csv"
'''

with open(os.path.join(sklearn_inference_dir, 'inference.py'), 'w') as f:
    f.write(sklearn_inference_script)

print(f"SKLearn inference script: {sklearn_inference_dir}")

SKLearn inference script: /home/model-server/tmp/tmph9qzd6q8


In [12]:
# Repack SKLearn model with inference code using repack_model utility
sklearn_repacked_uri = f"s3://{bucket}/{prefix}/sklearn/repacked/model.tar.gz"

repack_model(
    inference_script="inference.py",
    source_directory=sklearn_inference_dir,
    dependencies=[],
    model_uri=sklearn_model_uri,
    repacked_model_uri=sklearn_repacked_uri,
    sagemaker_session=sagemaker_session
)

print(f"Repacked SKLearn model: {sklearn_repacked_uri}")

Repacked SKLearn model: s3://sagemaker-us-west-2-674622101542/inference-pipeline-v3/5e35f9a2/sklearn/repacked/model.tar.gz


In [13]:
# XGBoost uses built-in inference - no custom script needed for basic CSV input/output
# The XGBoost container handles text/csv natively
xgboost_repacked_uri = xgboost_model_uri
print(f"XGBoost model (no repack needed): {xgboost_repacked_uri}")

XGBoost model (no repack needed): s3://sagemaker-us-west-2-674622101542/xgboost-classifier/xgboost-classifier-20260720071535/output/model.tar.gz


---
## Step 5: Deploy with ModelBuilder (Recommended)

`ModelBuilder` provides a simplified deployment experience for inference pipelines. This is the **recommended approach** for most use cases.

**How it works:**
1. Create individual `Model` objects using `Model.create()` with `primary_container`
2. Pass the list of models to `ModelBuilder(model=[model1, model2, ...])`
3. Call `build()` to create the pipeline model
4. Call `deploy()` to create the endpoint

**Note:** Each `Model` must use `primary_container` (not `containers`). ModelBuilder extracts the container definitions and combines them into a pipeline.

In [14]:
# Get inference images
sklearn_inference_image = retrieve(
    framework="sklearn", region=region, version="1.4-2"
)
xgboost_inference_image = retrieve(
    framework="xgboost", region=region, version="3.0-5"
)
print(f"SKLearn inference image: {sklearn_inference_image}")
print(f"XGBoost inference image: {xgboost_inference_image}")

[07/20/26 07:17:49] INFO     Defaulting to only available Python version: py3                     ]8;id=16060498;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=16060499;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/image_uris.py#615\615]8;;\

                    INFO     Defaulting to only supported image scope: cpu.                       ]8;id=16060504;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=16060505;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/image_uris.py#539\539]8;;\

                    INFO     Ignoring unnecessary instance type: None.                            ]8;id=16060510;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=16060511;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/image_uris.py#535\535]8;;\

SKLearn inference image: 246618743249.dkr.ecr.us-west-2.amazonaws.com/sagemaker-scikit-learn:1.4-2-cpu-py3
XGBoost inference image: 246618743249.dkr.ecr.us-west-2.amazonaws.com/sagemaker-xgboost:3.0-5


In [15]:
# Create individual Model objects for each container
sklearn_model_name = f"sklearn-model-{unique_id}"
xgboost_model_name = f"xgboost-model-{unique_id}"

# SKLearn preprocessing model
sklearn_model = Model.create(
    model_name=sklearn_model_name,
    primary_container=ContainerDefinition(
        image=sklearn_inference_image,
        model_data_url=sklearn_repacked_uri,
        environment={
            "SAGEMAKER_PROGRAM": "inference.py",
            "SAGEMAKER_SUBMIT_DIRECTORY": "/opt/ml/model/code"
        }
    ),
    execution_role_arn=role
)

# XGBoost inference model
xgboost_model = Model.create(
    model_name=xgboost_model_name,
    primary_container=ContainerDefinition(
        image=xgboost_inference_image,
        model_data_url=xgboost_repacked_uri
    ),
    execution_role_arn=role
)

print(f"Created sklearn model: {sklearn_model_name}")
print(f"Created xgboost model: {xgboost_model_name}")

                    INFO     Creating model resource.                                            ]8;id=16060517;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=16060518;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#20593\20593]8;;\

[07/20/26 07:17:50] INFO     Creating model resource.                                            ]8;id=16060523;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=16060524;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#20593\20593]8;;\

Created sklearn model: sklearn-model-5e35f9a2
Created xgboost model: xgboost-model-5e35f9a2


In [16]:
# Create ModelBuilder with list of Models for inference pipeline
pipeline_builder = ModelBuilder(
    model=[sklearn_model, xgboost_model],
    role_arn=role,
    sagemaker_session=sagemaker_session
)

# Build the pipeline model
pipeline_model_mb = pipeline_builder.build()
print(f"Pipeline model built: {pipeline_model_mb.model_name}")

[07/20/26 07:17:52] DEBUG    Auto-detecting optimal instance type for model...           ]8;id=16060531;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=16060532;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_builder_utils.py#341\341]8;;\

                    DEBUG    Using default CPU instance type: ml.m5.large                ]8;id=16060538;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=16060539;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_builder_utils.py#375\375]8;;\

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=16060544;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=16060545;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

[07/20/26 07:17:53] INFO     Cannot simulate policies for                                  ]8;id=16060550;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=16060551;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#363\363]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (access denied); permission verdict unknown.                                 

                    WARNING  Could not verify permissions for role                         ]8;id=16060556;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=16060557;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#585\585]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (caller lacks iam:SimulatePrincipalPolicy).                                  
                             Proceeding with it. If the operation later fails with an                              
                             access-denied error, ensure the role has the required                                 
                             permissions for 'serving' (see                                                        
                             IamRoleResolver().get_required_actions('serving')) or create                          
                             a dedicated role via                                                                  
                             IamRoleResolver().create_execution_role(role_type='serving').                         

...

                    WARNING  Could not verify permissions for role                         ]8;id=16060568;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=16060569;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#585\585]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (caller lacks iam:SimulatePrincipalPolicy).                                  
                             Proceeding with it. If the operation later fails with an                              
                             access-denied error, ensure the role has the required                                 
                             permissions for 'serving' (see                                                        
                             IamRoleResolver().get_required_actions('serving')) or create                          
                             a dedicated role via                                                                  
                             IamRoleResolver().create_execution_role(role_type='serving').                         

sagemaker.config INFO - Applied value(s) from config key = SageMaker.Model.Tags


[07/20/26 07:17:54] INFO     Creating model with name: model-dd69ecd4                        ]8;id=16060576;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/session_helper.py\session_helper.py]8;;\:]8;id=16060577;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/session_helper.py#1922\1922]8;;\

[07/20/26 07:17:55] INFO     ✅ Model has been created: 'model-dd69ecd4' using server None in ]8;id=16060584;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_builder.py\model_builder.py]8;;\:]8;id=16060585;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_builder.py#3656\3656]8;;\
                             SAGEMAKER_ENDPOINT mode (ARN:                                                         
                             arn:aws:sagemaker:us-west-2:674622101542:model/model-dd69ecd4)                        

Pipeline model built: model-dd69ecd4


In [17]:
# Deploy using ModelBuilder
endpoint_name_mb = f"pipeline-mb-{unique_id}"

endpoint_mb = pipeline_builder.deploy(
    endpoint_name=endpoint_name_mb,
    instance_type="ml.m5.large",
    initial_instance_count=1
)

print(f"Endpoint deployed: {endpoint_name_mb}")

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=16060590;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=16060591;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

sagemaker.config INFO - Applied value(s) from config key = SageMaker.EndpointConfig.Tags


[07/20/26 07:17:56] INFO     Creating endpoint-config with name pipeline-mb-5e35f9a2         ]8;id=16060597;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/session_helper.py\session_helper.py]8;;\:]8;id=16060598;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/session_helper.py#1093\1093]8;;\

[07/20/26 07:17:57] INFO     Creating endpoint with name pipeline-mb-5e35f9a2                ]8;id=16060604;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/session_helper.py\session_helper.py]8;;\:]8;id=16060605;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/session_helper.py#1125\1125]8;;\

sagemaker.config INFO - Applied value(s) from config key = SageMaker.Endpoint.Tags


                    WARNING  Failed to enable live logging: An error occurred                ]8;id=16060611;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/session_helper.py\session_helper.py]8;;\:]8;id=16060612;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/session_helper.py#2844\2844]8;;\
                             (AccessDeniedException) when calling the FilterLogEvents                              
                             operation: User:                                                                      
                             arn:aws:sts::674622101542:assumed-role/NotebookTestEngine-Proce                       
                             ssingJobRole/SageMaker is not authorized to perform:                                  
                             logs:FilterLogEvents on resource:                                                     
                             arn:aws:logs:us-west-2:674622101542:log-group:/aws/sagemaker/En                       
                             dpoints/pipeline-mb-5e35f9a2:log-stream: because no                                   
                             identity-based policy allows the logs:FilterLogEvents action.                         
                             Fallback to default logging...                                                        

[07/20/26 07:21:58] INFO     ✅ Deployment successful: Endpoint 'pipeline-mb-5e35f9a2' using  ]8;id=16060618;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_builder.py\model_builder.py]8;;\:]8;id=16060619;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_builder.py#2977\2977]8;;\
                             None in SAGEMAKER_ENDPOINT mode (ARN:                                                 
                             arn:aws:sagemaker:us-west-2:674622101542:endpoint/pipeline-mb-5e                      
                             35f9a2)                                                                               

Endpoint deployed: pipeline-mb-5e35f9a2


In [18]:
# Test the ModelBuilder-deployed endpoint
test_samples = test_df.iloc[:5, :4].values
test_labels = test_df.iloc[:5, 4].values

csv_data = "\n".join([",".join([str(x) for x in row]) for row in test_samples])

response = endpoint_mb.invoke(
    body=csv_data,
    content_type="text/csv",
    accept="text/csv"
)

result = response.body.read().decode('utf-8')
predictions = [float(x) for x in result.strip().split('\n')]

print("ModelBuilder Pipeline Results:")
print(f"Predictions: {predictions}")
print(f"Binary: {[1 if p > 0.5 else 0 for p in predictions]}")
print(f"Actual: {list(test_labels)}")

ModelBuilder Pipeline Results:
Predictions: [0.00037268511368893087, 0.0005569442291744053, 0.00037268511368893087, 0.00037268511368893087, 0.00037268511368893087]
Binary: [0, 0, 0, 0, 0]
Actual: [np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0)]


In [19]:
# Clean up ModelBuilder resources
try:
    endpoint_mb.delete()
    print(f"Deleted endpoint: {endpoint_name_mb}")
except Exception as e:
    print(f"Error: {e}")

try:
    sklearn_model.delete()
    xgboost_model.delete()
    pipeline_model_mb.delete()
    print("Deleted models")
except Exception as e:
    print(f"Error: {e}")

[07/20/26 07:21:59] INFO     Deleting Endpoint - pipeline-mb-5e35f9a2                            ]8;id=16060625;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=16060626;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#10428\10428]8;;\

Deleted endpoint: pipeline-mb-5e35f9a2


                    INFO     Deleting Model - sklearn-model-5e35f9a2                             ]8;id=16060632;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=16060633;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#20740\20740]8;;\

[07/20/26 07:22:01] INFO     Deleting Model - xgboost-model-5e35f9a2                             ]8;id=16060638;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=16060639;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#20740\20740]8;;\

[07/20/26 07:22:03] INFO     Deleting Model - model-dd69ecd4                                     ]8;id=16060644;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=16060645;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#20740\20740]8;;\

Deleted models


---
## Alternative: Low-level API Deployment

This section demonstrates the low-level approach using `Model.create()` with multiple `ContainerDefinition` objects. Use this when you need fine-grained control over the deployment configuration.

**Key parameters:**
- `containers`: List of `ContainerDefinition` objects executed in order
- `container_hostname`: Identifies each container in logs and metrics
- `inference_execution_config`: Set to `Serial` for pipeline execution
- `environment`: Must include `SAGEMAKER_PROGRAM` and `SAGEMAKER_SUBMIT_DIRECTORY` for custom inference scripts

In [20]:
# Create inference pipeline model
pipeline_model_name = f"pipeline-model-{unique_id}"

pipeline_model = Model.create(
    model_name=pipeline_model_name,
    containers=[
        ContainerDefinition(
            container_hostname="preprocessing",
            image=sklearn_inference_image,
            model_data_url=sklearn_repacked_uri,
            environment={
                "SAGEMAKER_PROGRAM": "inference.py",
                "SAGEMAKER_SUBMIT_DIRECTORY": "/opt/ml/model/code"
            }
        ),
        ContainerDefinition(
            container_hostname="inference",
            image=xgboost_inference_image,
            model_data_url=xgboost_repacked_uri
        )
    ],
    inference_execution_config=InferenceExecutionConfig(mode="Serial"),
    execution_role_arn=role
)

print(f"Pipeline model created: {pipeline_model.model_name}")

                    INFO     Creating model resource.                                            ]8;id=16060650;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=16060651;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#20593\20593]8;;\

Pipeline model created: pipeline-model-5e35f9a2


---
### Deploy the Inference Pipeline

Deployment requires creating an `EndpointConfig` and then an `Endpoint`. This is the low-level approach that gives you full control over the deployment configuration.

In [21]:
# Create endpoint config and endpoint
endpoint_config_name = f"pipeline-config-{unique_id}"
endpoint_name = f"pipeline-endpoint-{unique_id}"

endpoint_config = EndpointConfig.create(
    endpoint_config_name=endpoint_config_name,
    production_variants=[
        ProductionVariant(
            variant_name="AllTraffic",
            model_name=pipeline_model_name,
            initial_instance_count=1,
            instance_type="ml.m5.large"
        )
    ]
)

endpoint = Endpoint.create(
    endpoint_name=endpoint_name,
    endpoint_config_name=endpoint_config_name
)

print(f"Creating endpoint: {endpoint_name}")
endpoint.wait_for_status(target_status="InService")
print(f"Endpoint ready: {endpoint_name}")

[07/20/26 07:22:04] INFO     Creating endpoint_config resource.                                  ]8;id=16060657;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=16060658;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#11069\11069]8;;\

[07/20/26 07:22:05] INFO     Creating endpoint resource.                                         ]8;id=16060664;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=16060665;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#10228\10228]8;;\

Creating endpoint: pipeline-endpoint-5e35f9a2


[07/20/26 07:25:53] INFO     Final Resource Status: InService                                    ]8;id=16060671;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=16060672;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#10484\10484]8;;\

Endpoint ready: pipeline-endpoint-5e35f9a2


---
### Test the Inference Pipeline

When invoking the pipeline:
- Your input goes to Container 1 (SKLearn preprocessing)
- Container 1's output automatically flows to Container 2 (XGBoost)
- Container 2's output is returned as the final response

The `content_type` you specify applies to Container 1's input, and `accept` applies to Container 2's output.

In [22]:
# Test inference
test_samples = test_df.iloc[:5, :4].values
test_labels = test_df.iloc[:5, 4].values

csv_data = "\n".join([",".join([str(x) for x in row]) for row in test_samples])

response = endpoint.invoke(
    body=csv_data,
    content_type="text/csv",
    accept="text/csv"
)

result = response.body.read().decode('utf-8')
predictions = [float(x) for x in result.strip().split('\n')]

print("Pipeline Inference Results:")
print(f"Predictions (probabilities): {predictions}")
print(f"Binary predictions: {[1 if p > 0.5 else 0 for p in predictions]}")
print(f"Actual labels: {list(test_labels)}")

Pipeline Inference Results:
Predictions (probabilities): [0.00037268511368893087, 0.0005569442291744053, 0.00037268511368893087, 0.00037268511368893087, 0.00037268511368893087]
Binary predictions: [0, 0, 0, 0, 0]
Actual labels: [np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0)]


---
### Clean Up

Delete resources in reverse order of creation: Endpoint → EndpointConfig → Model.

In [23]:
# Clean up resources
print("Cleaning up...")

try:
    endpoint.delete()
    print(f"Deleted endpoint: {endpoint_name}")
except Exception as e:
    print(f"Error: {e}")

try:
    endpoint_config.delete()
    print(f"Deleted endpoint config: {endpoint_config_name}")
except Exception as e:
    print(f"Error: {e}")

try:
    pipeline_model.delete()
    print(f"Deleted model: {pipeline_model_name}")
except Exception as e:
    print(f"Error: {e}")

print("Cleanup completed!")

Cleaning up...


[07/20/26 07:25:54] INFO     Deleting Endpoint - pipeline-endpoint-5e35f9a2                      ]8;id=16060677;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=16060678;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#10428\10428]8;;\

Deleted endpoint: pipeline-endpoint-5e35f9a2


                    INFO     Deleting EndpointConfig - pipeline-config-5e35f9a2                  ]8;id=16060684;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=16060685;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#11220\11220]8;;\

Deleted endpoint config: pipeline-config-5e35f9a2


                    INFO     Deleting Model - pipeline-model-5e35f9a2                            ]8;id=16060690;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=16060691;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#20740\20740]8;;\

Deleted model: pipeline-model-5e35f9a2
Cleanup completed!


---
## Summary

This notebook demonstrated two approaches for deploying inference pipelines in SageMaker V3.

### Approach 1: ModelBuilder (Recommended)

| Step | API | Description |
|------|-----|-------------|
| Training | `ModelTrainer` | High-level training with `SourceCode` and `InputData` |
| Repacking | `repack_model()` | Adds inference code to model artifacts |
| Models | `Model.create(primary_container=...)` | Individual models per container |
| Deploy | `ModelBuilder(model=[...]).deploy()` | Single call for build + deploy |

### Approach 2: Low-level APIs (Full Control)

| Step | API | Description |
|------|-----|-------------|
| Training | `ModelTrainer` | Same as above |
| Repacking | `repack_model()` | Same as above |
| Model | `Model.create(containers=[...])` | Creates multi-container pipeline model |
| Deploy | `EndpointConfig` + `Endpoint` | Explicit endpoint configuration |

### Key Concepts

**Training vs Inference Code Separation:**
- Training scripts focus on model fitting
- Inference logic added via `repack_model`

**Pipeline Data Flow:**
- `content_type` in `invoke()` → applies to first container's input
- `accept` in `invoke()` → applies to last container's output
- Intermediate data: controlled by `output_fn` return value

### When to Use Each Approach

- **ModelBuilder**: Quick deployment, recommended for most use cases
- **Low-level APIs**: Fine-grained control over endpoint configuration